<a href="https://colab.research.google.com/github/Dracarys38/Machyne-navchanya/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0_%D1%80%D0%BE%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9610_%D0%9C%D0%9D%2C_%D0%9F%D0%BE%D1%81%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D1%8F%D0%BA_%D0%86%D0%B3%D0%BE%D1%80_%D0%A1%D0%B5%D1%80%D0%B3%D1%96%D0%B9%D0%BE%D0%B2%D0%B8%D1%87_%D0%A4%D0%86%D0%A2_3_15%2C_10_%D0%B2%D0%B0%D1%80%D1%96%D0%B0%D0%BD%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip uninstall -y numpy surprise scikit-surprise
# !pip install "numpy<2.0" scikit-surprise

In [2]:
import pandas as pd
from surprise import Dataset, Reader
from surprise import SVD, KNNBasic
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy
from surprise.model_selection import train_test_split

import sys
import io

In [3]:
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

--2026-04-14 18:18:51--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip.2’

ml-100k.zip.2       100%[===================>]   4.70M  11.1MB/s    in 0.4s    

2026-04-14 18:18:52 (11.1 MB/s) - ‘ml-100k.zip.2’ saved [4924029/4924029]

Archive:  ml-100k.zip
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.base         
  infl

In [4]:
import pandas as pd

data = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'])
data.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [5]:
reader = Reader(rating_scale=(1, 5))
data_surprise = Dataset.load_from_df(data[['user_id', 'movie_id', 'rating']], reader)

In [7]:
trainset, testset = train_test_split(data_surprise, test_size=0.2)

# 3. GridSearchCV для SVD
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3)
grid_search_svd.fit(data_surprise)

print("Best RMSE for SVD:", grid_search_svd.best_score['rmse'])
print("Best params for SVD:", grid_search_svd.best_params['rmse'])

# 4. GridSearchCV для KNNBasic
param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True, False]
    }
}

# Приглушення виводу similarity matrix
sys_stdout_backup = sys.stdout
sys.stdout = io.StringIO()

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3)
grid_search_knn.fit(data_surprise)

# Повернення нормального виводу
sys.stdout = sys_stdout_backup

print("Best RMSE for KNNBasic:", grid_search_knn.best_score['rmse'])
print("Best params for KNNBasic:", grid_search_knn.best_params['rmse'])

# 5. Найкращий SVD
best_svd = SVD(**grid_search_svd.best_params['rmse'])
best_svd.fit(trainset)
predictions_svd = best_svd.test(testset)
print("Test RMSE (SVD):", accuracy.rmse(predictions_svd))

# 6. Найкращий KNNBasic
best_knn = KNNBasic(**grid_search_knn.best_params['rmse'])
best_knn.fit(trainset)
predictions_knn = best_knn.test(testset)
print("Test RMSE (KNNBasic):", accuracy.rmse(predictions_knn))

KeyboardInterrupt: 

In [8]:
best_svd = grid_search_svd.best_estimator['mae']
best_knn = grid_search_knn.best_estimator['mae']

print(f"Найкращий MAE для SVD: {grid_search_svd.best_score['mae']}")
print(f"Найкращий MAE для KNNBasic: {grid_search_knn.best_score['mae']}")

# Вибір моделі з найменшим MAE
if grid_search_svd.best_score['mae'] < grid_search_knn.best_score['mae']:
    best_model = best_svd
    print("Вибраний алгоритм: SVD")
else:
    best_model = best_knn
    print("Вибраний алгоритм: KNNBasic")

AttributeError: 'GridSearchCV' object has no attribute 'best_estimator'

In [ ]:
best_model.fit(trainset)

# Беремо реального користувача з Вашого DataFrame
user_id = data['user_id'].iloc[0]

# Якщо потрібно, дивимось його оцінки у trainset
inner_uid = trainset.to_inner_uid(user_id)
user_ratings = trainset.ur[inner_uid]
print(f"Кількість рецензій користувача {user_id}: {len(user_ratings)}")

# Знаходимо фільми, які користувач ще не оцінював
all_items = set(trainset.all_items())
rated_items = set([item for (item, _) in user_ratings])
unrated_items = all_items - rated_items

# Прогнозуємо рейтинги для неоцінених фільмів
predictions = [
    (item, best_model.predict(user_id, trainset.to_raw_iid(item)).est)
    for item in unrated_items
]

predictions.sort(key=lambda x: x[1], reverse=True)

# Виводимо топ-10 рекомендацій
print("Топ-10 фільмів, рекомендованих для користувача:")
for item_id, rating in predictions[:10]:
    print(f"Фільм {trainset.to_raw_iid(item_id)} з прогнозованим рейтингом {rating:.2f}")